In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [12]:
dataset= generate_dataset()
print(dataset)

[{'task': 'Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens only, must start and end with letter or number)'}, {'task': 'Create a JSON CloudFormation template snippet that defines an AWS IAM role with a trust relationship allowing EC2 instances to assume it'}, {'task': 'Write a regular expression that matches valid AWS ARN (Amazon Resource Name) format: arn:partition:service:region:account-id:resource-type/resource-id'}]


In [13]:
with open('dataset.json','w') as f:
    json.dump(dataset,f,indent=2)
    

In [ ]:
def run_prompt(test_case):
    """Meges the prompt and text case input , then returns the result"""
    prompt= """Please solve the following task 
    {test_case["task"]}
    * Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
    """
    messages=[]
    add_user_message(messages,prompt)
    add_assistant_message(messages,"```code")
    output=chat(messages)
    return output

In [33]:
def grade_by_model(test_case,output):
    # Create evaluation prompt
    eval_prompt =f""" You are an expert code reviewer . Evaluate thsi AI generated solution
Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>


    Output Format
    Provide your evalutions as a structured JSON object with :
    -"strengths": An array of 1-3 key strengths
    -"weakness": An array of 1-3 key areas for improvement
    -"reasoning": A concise explanation of your assessment
    -"score": A number between 1-10
"""
    messages = []
    add_user_message(messages,eval_prompt)
    add_assistant_message(messages,"```json")
    eval_text= chat(messages,stop_sequences=["```"])
    return json.loads(eval_text)


In [ ]:
import re 
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
def grade_syntax(response,test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt , then geades the result"""
    output = run_prompt(test_case)

    #Todo- Grading
    model_grade=grade_by_model(test_case,output)
    model_score= model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_syntax(output, test_case)
    
    score = (model_score + syntax_score) / 2
    return{
        "output":output,
        "test_case": test_case,
        "score": score,
        "reasoning" : reasoning
    }

In [40]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    average_score = mean([result["score"]  for result in results])
    print(f"Average score is is {average_score}")
    return results


In [41]:
with open("dataset.json","r") as f:
    dataset= json.load(f)
results = run_eval(dataset)
print(json.dumps(results,indent=2))


Average score is is 1.3333333333333333
[
  {
    "output": "I'd be happy to help! However, I don't see a task in your message. The `test_case[\"task\"]` appears to be a placeholder that wasn't filled in with actual content.\n\nCould you please provide:\n1. The specific task or problem you'd like me to solve\n2. Any relevant details, context, or constraints\n3. The expected format for the solution\n\nOnce you share the task details, I'll do my best to help you solve it!",
    "test_case": {
      "task": "Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens only, must start and end with letter or number)"
    },
    "score": 1,
    "reasoning": "This is not a valid solution to the stated problem. The task explicitly asked for a Python function to validate S3 bucket names with specific rules (3-63 characters, lowercase letters, numbers, hyphens only, must start/end with letter or number). Instead,

Given the model grader more context on what a good solution looks like 
Step 1 Uppdate the dataset geenration prompt to ask fro some solution criteria to be included for each test case 
Step 2 Update the grade_by_model prompt to include the solution criteria
